# 2. OpenRouter student experiments

Run the same conditions used by the earlier GPQA experiment: 0–8 plus condition 20, the no-external-teaching CoT baseline. Each condition receives its own resumable JSONL file under `pipeline_runs/`. Successful rows are skipped and failed rows are retried by default.

In [ ]:
from pathlib import Path
import importlib
import sys
EXPERIMENT_ROOT = Path.cwd()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    EXPERIMENT_ROOT = (Path.cwd() / 'other_experiments' / 'MMLU data').resolve()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or its own folder')
sys.path.insert(0, str(EXPERIMENT_ROOT)) if str(EXPERIMENT_ROOT) not in sys.path else None
import mmlu_common
import openrouter_experiments
importlib.reload(mmlu_common)
importlib.reload(openrouter_experiments)
print('Experiment root:', EXPERIMENT_ROOT)

## Settings
Edit the model and provider. Set `REASONING_ENABLED=False` for the direct/no-thinking comparison used in the earlier experiments.

In [ ]:
STUDENT_MODEL = 'qwen/qwen3-8b'
PROVIDERS = None                    # Or an ordered list such as ['provider-a/fp8', 'provider-b/fp8']
TARGET_CONDITIONS = list(range(9)) + [20]
NUM_ROWS = None                    # Per condition; None = every prepared question
START_ROW = 0
CONCURRENCY = 20
TEMPERATURE = 0.0
MAX_TOKENS = 4096
REASONING_ENABLED = False
REASONING_EFFORT = 'low'

## Run or resume
This cell makes OpenRouter API calls.

In [ ]:
results = openrouter_experiments.run_openrouter_experiments(
    student_model=STUDENT_MODEL,
    provider=PROVIDERS,
    condition_ids=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    concurrency=CONCURRENCY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    reasoning_enabled=REASONING_ENABLED,
    reasoning_effort=REASONING_EFFORT,
    retry_failed=True,
)
results

## Compact comparison

In [ ]:
import pandas as pd
pd.DataFrame.from_dict(results, orient='index')[
    ['requested', 'successful', 'failed', 'correct', 'accuracy', 'output_file']
].rename_axis('condition_id')